In [1]:
import numpy as np
from tabpfn import TabPFNClassifier
from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier

In [2]:
file = "/home/server/Projects/data/AKI/tabular_combined.npz"

# Load data
with np.load(file, allow_pickle=True) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_binary_train = data["y_binary_train"]
    y_binary_test = data["y_binary_test"]

In [3]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [3]:
from tabpfn_extensions.rf_pfn import RandomForestTabPFNClassifier
from tabpfn_extensions import TabPFNClassifier

clf_base = TabPFNClassifier(
    ignore_pretraining_limits=True,
    inference_config = {"SUBSAMPLE_SAMPLES": 2000} # Needs to be set low so that not OOM on fitting intermediate nodes
)

tabpfn_tree_clf = RandomForestTabPFNClassifier(
    tabpfn=clf_base,
    verbose=1,
    max_predict_time=60, # Will fit for one minute
    fit_nodes=True, # Wheather or not to fit intermediate nodes
    adaptive_tree=True, # Whather or not to validate if adding a leaf helps or not
    n_jobs=1
  )

In [ ]:

from sklearn.model_selection import cross_val_score, KFold

cv = KFold(random_state=42, n_splits=3, shuffle=True)
scores_raw_class = cross_val_score(tabpfn_tree_clf, X_train, y_binary_train, cv=cv, scoring='roc_auc', verbose=1)
scores_class = scores_raw_class.mean()

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   35.6s


Starting tree leaf validation train (47018, 515) valid (11755, 515)
Estimators: 1, Nodes per estimator: 31
Leaf: 0 Shape: (47018, 515) Shape test: 11755 / (11755, 515) N classes (Train Test) 2 2 /  2


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 47018 is greater than the maximum Number of samples 10000 supp

In [6]:
scores_raw_class

array([nan, nan, nan])

In [7]:
tabpfn_subsample_clf = TabPFNClassifier(
    ignore_pretraining_limits=True,  # (bool) Allows the use of datasets larger than pretraining limits.
    n_estimators=32,                 # (int) Number of estimators for ensembling; improves accuracy with higher values.
    inference_config={
        "SUBSAMPLE_SAMPLES": 2000  # (int) Maximum number of samples per inference step to manage memory usage.
    },
)

In [8]:
cv = KFold(random_state=42, n_splits=3, shuffle=True)
scores_raw_class = cross_val_score(tabpfn_subsample_clf, X_train, y_binary_train, cv=cv, scoring='roc_auc', verbose=1)
scores_class = scores_raw_class.mean()

/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 58773 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:1000: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/

# Justin's Attempt
## RF Preprocessing

Broken, trains but takes forever

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import time
import torch # For checking CUDA availability and TabPFN

# Imports for RandomForestTabPFNClassifier
from tabpfn import TabPFNClassifier as BaseTabPFNClassifier # The base classifier for RF-TabPFN
from tabpfn_extensions.rf_pfn import RandomForestTabPFNClassifier

# Load data
# Assuming the npz file contains NumPy arrays directly
with np.load('/home/server/Projects/data/AKI/preop_trainable/unfiltered.npz', allow_pickle=True) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_binary_train = data["y_binary_train"]
    y_binary_test = data["y_binary_test"]

print(f"Data loaded: X_train shape: {X_train.shape}, y_train shape: {y_binary_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_binary_test.shape}")


print("-" * 50)
print("\nTraining and Evaluating RandomForestTabPFN Classifier for larger data...")
print("-" * 50)

# -------------------- Device Configuration --------------------
if torch.cuda.is_available():
    device = 'cuda'
    print("CUDA (GPU) is available. Using GPU.")
else:
    device = 'cpu'
    print("CUDA (GPU) not available. Using CPU. This might be slow for larger datasets.")

# -------------------- TabPFN Configuration for Larger Datasets --------------------
# Strategy: RandomForestTabPFNClassifier
# This uses a Random Forest structure where TabPFN is applied.
# It's generally more suited for datasets larger than TabPFN's typical pretraining limits.

# Configure the base TabPFN classifier that will be used within the Random Forest structure
# ignore_pretraining_limits=True allows usage on larger datasets.
# SUBSAMPLE_SAMPLES helps manage memory by subsampling data for TabPFN's inference at each node/leaf.
clf_base = BaseTabPFNClassifier(
    device=device,
    ignore_pretraining_limits=True,
    # Adjust SUBSAMPLE_SAMPLES based on your dataset size and available memory.
    # The demo script uses 10000.
    inference_config={"SUBSAMPLE_SAMPLES": min(X_train.shape[0], 10000)}
)

# Configure the RandomForestTabPFNClassifier
# max_predict_time: Limits the time spent fitting the forest.
# fit_nodes: Whether to fit TabPFN on intermediate nodes of the trees.
# adaptive_tree: Whether to validate if adding a leaf helps.
tabpfn_classifier = RandomForestTabPFNClassifier(
    tabpfn=clf_base,
    verbose=1, # Set to 0 for less output
    max_predict_time=120, # Increased from demo for potentially larger data, adjust as needed
    fit_nodes=True,
    adaptive_tree=True,
    random_state=42 # For reproducibility
  )

# Alternative Strategy: Subsampled Ensemble using TabPFNClassifier from tabpfn_extensions
# You can uncomment and try this if RandomForestTabPFNClassifier is not suitable or for comparison.
# from tabpfn_extensions import TabPFNClassifier as ExtendedTabPFNClassifier
# tabpfn_classifier = ExtendedTabPFNClassifier(
#     device=device,
#     ignore_pretraining_limits=True,
#     n_estimators=32, # Number of estimators for ensembling
#     inference_config={
#         "SUBSAMPLE_SAMPLES": min(X_train.shape[0], 10000) # Max samples per inference step
#     },
# )


try:
    print(f"\nTraining {type(tabpfn_classifier).__name__}...")
    start_time_tabpfn = time.time()

    # RandomForestTabPFNClassifier expects NumPy arrays.
    tabpfn_classifier.fit(X_train, y_binary_train)
    end_time_tabpfn = time.time()
    print(f"TabPFN fitting completed in {end_time_tabpfn - start_time_tabpfn:.2f} seconds.")

    # -------------------- Make Predictions with TabPFN --------------------
    print("\nEvaluating TabPFN Model...")

    # Predict binary labels
    y_pred_tabpfn = tabpfn_classifier.predict(X_test)

    # Predict probabilities for ROC Curve
    y_prob_tabpfn = tabpfn_classifier.predict_proba(X_test)[:, 1] # Probability of class 1

    # -------------------- Evaluate TabPFN Model Performance --------------------
    accuracy_tabpfn = accuracy_score(y_binary_test, y_pred_tabpfn)
    print(f'TabPFN Accuracy: {accuracy_tabpfn:.2f}')

    cm_tabpfn = confusion_matrix(y_binary_test, y_pred_tabpfn)
    print('TabPFN Confusion Matrix:')
    print(cm_tabpfn)

    report_tabpfn = classification_report(y_binary_test, y_pred_tabpfn)
    print('TabPFN Classification Report:')
    print(report_tabpfn)

    # Compute ROC curve for TabPFN
    fpr_tabpfn, tpr_tabpfn, _ = roc_curve(y_binary_test, y_prob_tabpfn)
    roc_auc_tabpfn = auc(fpr_tabpfn, tpr_tabpfn)

    # Plot ROC Curve for TabPFN
    plt.figure()
    plt.plot(fpr_tabpfn, tpr_tabpfn, color='blue', lw=2, label=f'TabPFN ROC curve (area = {roc_auc_tabpfn:.2f})')
    plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{type(tabpfn_classifier).__name__}: ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

except ImportError as e:
    print(f"ImportError: {e}")
    print("Please ensure TabPFN, tabpfn_extensions, and PyTorch are correctly installed.")
    print("For tabpfn_extensions, you might need to clone the repo and install editable:")
    print("  git clone https://github.com/PriorLabs/tabpfn-extensions")
    print("  pip install -e tabpfn-extensions")
except Exception as e:
    print(f"An error occurred with TabPFN: {e}")
    print("Ensure you have PyTorch installed. Using a GPU is recommended for performance.")
    print("If using RandomForestTabPFNClassifier, check the parameters like 'max_predict_time' and 'SUBSAMPLE_SAMPLES'.")

Data loaded: X_train shape: (76768, 79), y_train shape: (76768,)
X_test shape: (10255, 79), y_test shape: (10255,)
--------------------------------------------------

Training and Evaluating RandomForestTabPFN Classifier for larger data...
--------------------------------------------------
CUDA (GPU) is available. Using GPU.

Training RandomForestTabPFNClassifier...


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   21.9s


TabPFN fitting completed in 44.76 seconds.

Evaluating TabPFN Model...
Starting tree leaf validation train (61414, 79) valid (15354, 79)
Estimators: 1, Nodes per estimator: 45
Leaf: 0 Shape: (61414, 79) Shape test: 15354 / (15354, 79) N classes (Train Test) 2 2 /  2


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 61414 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(


: 

## ExtendedTabPFN + Subsampling

Takes way too long to run

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import time
import torch

# Import ExtendedTabPFNClassifier from tabpfn_extensions
from tabpfn_extensions import TabPFNClassifier as ExtendedTabPFNClassifier

# Load data
with np.load('/home/server/Projects/data/AKI/preop_trainable/unfiltered.npz', allow_pickle=True) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_binary_train = data["y_binary_train"]
    y_binary_test = data["y_binary_test"]

print(f"Data loaded: X_train shape: {X_train.shape}, y_train shape: {y_binary_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_binary_test.shape}")

print("-" * 50)
print("\nTraining and Evaluating ExtendedTabPFNClassifier (Ensemble with Subsampling)...")
print("-" * 50)

# -------------------- Device Configuration --------------------
if torch.cuda.is_available():
    device = 'cuda'
    print("CUDA (GPU) is available. Using GPU.")
else:
    device = 'cpu'
    print("CUDA (GPU) not available. Using CPU. This might be slow for larger datasets.")

# -------------------- ExtendedTabPFNClassifier Configuration --------------------
# This classifier is an ensemble of TabPFN models and allows direct inference_config.

# Number of TabPFN models in the ensemble
N_ESTIMATORS_CONFIG = 32  # You can adjust this; more estimators might improve performance but increase time.

# SUBSAMPLE_SAMPLES for each base TabPFN model in the ensemble.
# This is crucial for memory management. Start with a known safe value.
SUBSAMPLE_SAMPLES_CONFIG = 512 # Adjust if OOM: 256, 1024, etc.
print(f"ExtendedTabPFNClassifier will use {N_ESTIMATORS_CONFIG} estimators.")
print(f"Each estimator will use SUBSAMPLE_SAMPLES: {SUBSAMPLE_SAMPLES_CONFIG}")

tabpfn_classifier = ExtendedTabPFNClassifier(
    device=device,
    ignore_pretraining_limits=True,  # Allows base TabPFN to use more samples/features
    n_estimators=N_ESTIMATORS_CONFIG,     # Number of TabPFN models in the ensemble
    inference_config={                 # Config for each base TabPFN model
        "SUBSAMPLE_SAMPLES": SUBSAMPLE_SAMPLES_CONFIG
    },
    random_state=42                    # For reproducibility
)

try:
    print(f"\nTraining {type(tabpfn_classifier).__name__}...")
    start_time_tabpfn = time.time()

    tabpfn_classifier.fit(X_train, y_binary_train)
    end_time_tabpfn = time.time()
    print(f"Fitting completed in {end_time_tabpfn - start_time_tabpfn:.2f} seconds.")

    # -------------------- Make Predictions --------------------
    print("\nEvaluating Model...")
    y_pred_tabpfn = tabpfn_classifier.predict(X_test)
    y_prob_tabpfn = tabpfn_classifier.predict_proba(X_test)[:, 1] # Probability of class 1

    # -------------------- Evaluate Model Performance --------------------
    accuracy_tabpfn = accuracy_score(y_binary_test, y_pred_tabpfn)
    print(f'Accuracy: {accuracy_tabpfn:.2f}')

    cm_tabpfn = confusion_matrix(y_binary_test, y_pred_tabpfn)
    print('Confusion Matrix:')
    print(cm_tabpfn)

    report_tabpfn = classification_report(y_binary_test, y_pred_tabpfn)
    print('Classification Report:')
    print(report_tabpfn)

    # Compute ROC curve
    fpr_tabpfn, tpr_tabpfn, _ = roc_curve(y_binary_test, y_prob_tabpfn)
    roc_auc_tabpfn = auc(fpr_tabpfn, tpr_tabpfn)

    # Plot ROC Curve
    plt.figure()
    plt.plot(fpr_tabpfn, tpr_tabpfn, color='orange', lw=2, label=f'ROC curve (area = {roc_auc_tabpfn:.2f})')
    plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{type(tabpfn_classifier).__name__}: ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

except ImportError as e:
    print(f"ImportError: {e}")
    print("Please ensure TabPFN and tabpfn_extensions are correctly installed.")
except torch.cuda.OutOfMemoryError as e:
    print(f"CUDA Out of Memory with {type(tabpfn_classifier).__name__}: {e}")
    print("Suggestions:")
    print(f"1. Try further reducing SUBSAMPLE_SAMPLES_CONFIG (current: {SUBSAMPLE_SAMPLES_CONFIG}). E.g., 256 or 128.")
    print(f"2. Reduce N_ESTIMATORS_CONFIG (current: {N_ESTIMATORS_CONFIG}) if many estimators are causing cumulative memory pressure (less likely than SUBSAMPLE_SAMPLES_CONFIG being too high).")
    print(f"3. Ensure the PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True environment variable is set before running.")
except Exception as e:
    print(f"An error occurred with {type(tabpfn_classifier).__name__}: {e}")

Data loaded: X_train shape: (76768, 79), y_train shape: (76768,)
X_test shape: (10255, 79), y_test shape: (10255,)
--------------------------------------------------

Training and Evaluating ExtendedTabPFNClassifier (Ensemble with Subsampling)...
--------------------------------------------------
CUDA (GPU) is available. Using GPU.
ExtendedTabPFNClassifier will use 32 estimators.
Each estimator will use SUBSAMPLE_SAMPLES: 512

Training TabPFNClassifier...


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 76768 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(


Fitting completed in 4.73 seconds.

Evaluating Model...


## Ensemble + Subsampling

Broken - OOM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import time
import torch

# Import for AutoTabPFNClassifier which handles post-hoc ensembling
from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier

# Load data
with np.load('/home/server/Projects/data/AKI/preop_trainable/unfiltered.npz', allow_pickle=True) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_binary_train = data["y_binary_train"]
    y_binary_test = data["y_binary_test"]

print(f"Data loaded: X_train shape: {X_train.shape}, y_train shape: {y_binary_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_binary_test.shape}")

print("-" * 50)
print("\nTraining and Evaluating AutoTabPFNClassifier (with Post-Hoc Ensembling and Memory Control)...")
print("-" * 50)

# -------------------- Device Configuration --------------------
if torch.cuda.is_available():
    device = 'cuda'
    print("CUDA (GPU) is available. Using GPU.")
else:
    device = 'cpu'
    print("CUDA (GPU) not available. Using CPU. This might be slow for larger datasets.")

# -------------------- AutoTabPFNClassifier Configuration --------------------
TIME_LIMIT_FOR_AUTO_ENSEMBLE = 300 # Time in seconds (e.g., 5 minutes). Adjust as needed.

# Define SUBSAMPLE_SAMPLES for base TabPFN models used by AutoTabPFNClassifier
# Start with a value known to be safe or very conservative for your GPU
SUBSAMPLE_SAMPLES_CONFIG = 512 # Or 1024, 2048. Adjust based on your VRAM.
print(f"AutoTabPFNClassifier max_time: {TIME_LIMIT_FOR_AUTO_ENSEMBLE} seconds.")
print(f"Base TabPFN models within AutoTabPFN will use SUBSAMPLE_SAMPLES: {SUBSAMPLE_SAMPLES_CONFIG}")

tabpfn_classifier = AutoTabPFNClassifier(
    device=device, # Sets device for AutoTabPFN and should propagate to base models
    ignore_pretraining_limits=True,  # Allows base TabPFN to use more samples/features than default
    max_time=TIME_LIMIT_FOR_AUTO_ENSEMBLE, # Budget for HPO and ensembling
    random_state=42, # For reproducibility
    phe_init_args={ # Arguments for the internal AutoPostHocEnsemblePredictor
        "base_model_kwargs": { # Keyword arguments for the base TabPFNClassifier instances
            "inference_config": {"SUBSAMPLE_SAMPLES": SUBSAMPLE_SAMPLES_CONFIG}
            # 'device' and 'ignore_pretraining_limits' are set at AutoTabPFNClassifier level
            # and are expected to be handled correctly for the base models.
            # If not, you could also add them here:
            # "device": device,
            # "ignore_pretraining_limits": True
        }
    }
    # verbose parameter is not directly supported by AutoTabPFNClassifier
)

try:
    print(f"\nTraining {type(tabpfn_classifier).__name__}...")
    print(f"This might take up to {TIME_LIMIT_FOR_AUTO_ENSEMBLE / 60:.1f} minutes due to HPO and ensembling...")
    start_time_tabpfn = time.time()

    tabpfn_classifier.fit(X_train, y_binary_train)
    end_time_tabpfn = time.time()
    print(f"AutoTabPFN fitting completed in {end_time_tabpfn - start_time_tabpfn:.2f} seconds.")

    # -------------------- Make Predictions with AutoTabPFN --------------------
    print("\nEvaluating AutoTabPFN Model...")

    y_pred_tabpfn = tabpfn_classifier.predict(X_test)
    y_prob_tabpfn = tabpfn_classifier.predict_proba(X_test)[:, 1]

    # -------------------- Evaluate Model Performance --------------------
    accuracy_tabpfn = accuracy_score(y_binary_test, y_pred_tabpfn)
    print(f'AutoTabPFN Accuracy: {accuracy_tabpfn:.2f}')

    cm_tabpfn = confusion_matrix(y_binary_test, y_pred_tabpfn)
    print('AutoTabPFN Confusion Matrix:')
    print(cm_tabpfn)

    report_tabpfn = classification_report(y_binary_test, y_pred_tabpfn)
    print('AutoTabPFN Classification Report:')
    print(report_tabpfn)

    fpr_tabpfn, tpr_tabpfn, _ = roc_curve(y_binary_test, y_prob_tabpfn)
    roc_auc_tabpfn = auc(fpr_tabpfn, tpr_tabpfn)

    plt.figure()
    plt.plot(fpr_tabpfn, tpr_tabpfn, color='purple', lw=2, label=f'AutoTabPFN ROC (area = {roc_auc_tabpfn:.2f})')
    plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{type(tabpfn_classifier).__name__}: ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

except ImportError as e:
    print(f"ImportError: {e}")
    print("Please ensure TabPFN, tabpfn_extensions (with post_hoc_ensembles), and PyTorch are correctly installed.")
except torch.cuda.OutOfMemoryError as e:
    print(f"CUDA Out of Memory with {type(tabpfn_classifier).__name__}: {e}")
    print("Suggestions:")
    print(f"1. Try further reducing SUBSAMPLE_SAMPLES_CONFIG (current: {SUBSAMPLE_SAMPLES_CONFIG}). Values like 256 or 128 might be needed for very limited VRAM.")
    print(f"2. Ensure the PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True environment variable is set before running.")
    print(f"3. If issues persist, the combination of AutoTabPFNClassifier's ensembling and your dataset size might still be too much for the available VRAM, even with subsampling. Consider using the simpler ExtendedTabPFNClassifier with explicit subsampling control, which you had working before.")
except Exception as e:
    print(f"An error occurred with {type(tabpfn_classifier).__name__}: {e}")

Data loaded: X_train shape: (76768, 79), y_train shape: (76768,)
X_test shape: (10255, 79), y_test shape: (10255,)
--------------------------------------------------

Training and Evaluating AutoTabPFNClassifier (with Post-Hoc Ensembling and Memory Control)...
--------------------------------------------------
CUDA (GPU) is available. Using GPU.
AutoTabPFNClassifier max_time: 300 seconds.
Base TabPFN models within AutoTabPFN will use SUBSAMPLE_SAMPLES: 512

Training AutoTabPFNClassifier...
This might take up to 5.0 minutes due to HPO and ensembling...
An error occurred with AutoTabPFNClassifier: AutoPostHocEnsemblePredictor.__init__() got an unexpected keyword argument 'base_model_kwargs'


: 